# AI401 — Expert Systems Lab
## Experiment 2: Production Rules and the Match Phase

**Aim:**  
To implement the Match Phase of the Recognize-Act Cycle in a production rule-based expert system, support positive and negated conditions (`NOT fact`), implement refractoriness to prevent infinite loops, compute conflict sets dynamically, and benchmark match phase complexity against the RETE algorithm.


---
### Core Implementation: Rule and ProductionSystemMatcher
The `Rule` class represents IF-THEN rules. The `ProductionSystemMatcher` matches conditions against Working Memory (WM), checking refractoriness and handling negation.


In [ ]:
import time
import random

class Rule:
    """
    Represents an IF-THEN production rule.
    - id: Unique rule identifier (e.g., 'R1')
    - conditions: List of condition strings (e.g., ['fever', 'cough'] or ['fever', 'NOT flu'])
    - conclusion: Fact derived when the rule fires (e.g., 'viral_infection')
    """
    def __init__(self, rule_id: str, conditions: list, conclusion: str):
        self.id = rule_id
        self.conditions = conditions
        self.conclusion = conclusion

    def __repr__(self):
        cond_str = " AND ".join(self.conditions)
        return f"[{self.id}] IF {cond_str} THEN {self.conclusion}"


class ProductionSystemMatcher:
    """
    Implements the Match Phase of the Recognize-Act Cycle.
    Matches rules against Working Memory (WM) to construct the Conflict Set.
    """
    def __init__(self, rules: list):
        self.rules = rules

    def is_rule_applicable(self, rule: Rule, working_memory: set) -> bool:
        """
        Determines whether a rule is applicable:
        1. All positive conditions must be in working_memory.
        2. All negated conditions ('NOT fact') must be ABSENT from working_memory.
        3. Refractoriness: Rule conclusion must NOT already be in working_memory.
        """
        # Refractoriness check: prevent rule from firing if conclusion already exists
        if rule.conclusion in working_memory:
            return False

        for condition in rule.conditions:
            if condition.startswith("NOT "):
                negated_fact = condition[4:].strip()
                if negated_fact in working_memory:
                    return False
            else:
                if condition not in working_memory:
                    return False

        return True

    def compute_conflict_set(self, working_memory: set) -> list:
        """Collects all rules that currently match working memory."""
        conflict_set = []
        for rule in self.rules:
            if self.is_rule_applicable(rule, working_memory):
                conflict_set.append(rule)
        return conflict_set


---
### Part 1: Medical Diagnosis Rule Base Demonstration
Evaluating conflict sets against different Working Memory states.


In [ ]:
# Define Rule Base for Medical Diagnosis
rule_base = [
    Rule("R1", ["fever", "cough"], "viral_infection"),
    Rule("R2", ["viral_infection", "runny_nose"], "common_cold"),
    Rule("R3", ["fever", "spots"], "measles"),
    Rule("R4", ["fever", "stiff_neck"], "meningitis"),
    Rule("R5", ["cough", "chest_pain"], "pneumonia"),
]

print("=" * 70)
print("PART 1: PRODUCTION RULES & MATCH PHASE DEMONSTRATION")
print("=" * 70)

print("\n1. Defined Rule Base:")
for rule in rule_base:
    print(f"   {rule}")

matcher = ProductionSystemMatcher(rule_base)

# Test Case A
wm_a = {"fever", "cough"}
cs_a = matcher.compute_conflict_set(wm_a)
print(f"\n2. Working Memory: {sorted(list(wm_a))}")
print("   Conflict Set (Applicable Rules):")
for r in cs_a:
    print(f"     -> {r}")

# Test Case B (Adding runny_nose and viral_infection)
wm_b = {"fever", "cough", "runny_nose", "viral_infection"}
cs_b = matcher.compute_conflict_set(wm_b)
print(f"\n3. Working Memory: {sorted(list(wm_b))}")
print("   Conflict Set (Applicable Rules):")
for r in cs_b:
    print(f"     -> {r}")


---
### Exercise 1: Dynamic Conflict Set Tracking (Step-by-Step Fact Insertion)
Tracking how the conflict set dynamically evolves as individual symptoms are added to Working Memory.


In [ ]:
print("=" * 70)
print("EXERCISE 1: DYNAMIC CONFLICT SET TRACKING")
print("=" * 70)

rule_base_ex1 = [
    Rule("R1", ["fever"], "check_infection"),
    Rule("R2", ["fever", "cough"], "viral_infection"),
    Rule("R3", ["fever", "cough", "spots"], "measles"),
    Rule("R4", ["measles"], "quarantine_patient"),
]

matcher_ex1 = ProductionSystemMatcher(rule_base_ex1)
working_memory = set()

facts_to_add = ["fever", "cough", "spots", "measles"]

for step, fact in enumerate(facts_to_add, start=1):
    working_memory.add(fact)
    conflict_set = matcher_ex1.compute_conflict_set(working_memory)
    print(f"\nStep {step}: Added fact '{fact}'")
    print(f"  Working Memory: {sorted(list(working_memory))}")
    print(f"  Conflict Set Size: {len(conflict_set)}")
    print("  Conflict Set Rules:")
    if conflict_set:
        for r in conflict_set:
            print(f"     -> {r}")
    else:
        print("     -> [Empty Set]")


---
### Exercise 2: Extended Matcher with Negated Conditions ('NOT fact')
Demonstrates how rules activate and deactivate based on the presence or absence of negated facts.


In [ ]:
print("=" * 70)
print("EXERCISE 2: EXTENDED MATCHER WITH NEGATED CONDITIONS ('NOT fact')")
print("=" * 70)

rule_base_neg = [
    Rule("R1", ["fever", "NOT flu"], "suspect_measles"),
    Rule("R2", ["fever", "flu"], "treat_flu"),
    Rule("R3", ["cough", "NOT fever"], "allergic_cough"),
]

matcher_neg = ProductionSystemMatcher(rule_base_neg)

print("\nRule Base with Negated Conditions:")
for rule in rule_base_neg:
    print(f"   {rule}")

# Case 1: Fever present, flu absent -> R1 matches
wm1 = {"fever"}
cs1 = matcher_neg.compute_conflict_set(wm1)
print(f"\nCase 1 | WM = {sorted(list(wm1))}")
print("  Conflict Set:")
for r in cs1:
    print(f"     -> {r}")

# Case 2: Adding 'flu' to WM -> R1 no longer matches, R2 matches
wm2 = {"fever", "flu"}
cs2 = matcher_neg.compute_conflict_set(wm2)
print(f"\nCase 2 | WM = {sorted(list(wm2))} ('flu' added)")
print("  Conflict Set:")
for r in cs2:
    print(f"     -> {r}")

# Case 3: Cough present, fever absent -> R3 matches
wm3 = {"cough"}
cs3 = matcher_neg.compute_conflict_set(wm3)
print(f"\nCase 3 | WM = {sorted(list(wm3))}")
print("  Conflict Set:")
for r in cs3:
    print(f"     -> {r}")


---
### Exercise 3: Performance Scaling & RETE Algorithm Analysis
Benchmarking the naive matcher over 1,000 rules across varying WM sizes to demonstrate why Rete networks are necessary for scalable production systems.


In [ ]:
print("=" * 70)
print("EXERCISE 3: PERFORMANCE SCALING & RETE ALGORITHM ANALYSIS")
print("=" * 70)

random.seed(42)
fact_pool = [f"fact_{i}" for i in range(1, 201)]

# Generate 1,000 random rules
large_rule_base = []
for i in range(1, 1001):
    num_conds = random.randint(1, 4)
    conds = random.sample(fact_pool, num_conds)
    if random.random() < 0.1 and len(conds) > 1:
        conds[0] = f"NOT {conds[0]}"
    conclusion = f"derived_fact_{i}"
    large_rule_base.append(Rule(f"R{i}", conds, conclusion))

matcher_large = ProductionSystemMatcher(large_rule_base)

wm_sizes = [10, 50, 100, 200, 500]
print(f"Generated Rule Base Size: {len(large_rule_base)} rules\n")
print(f"{'WM Size':<10} | {'Conflict Set Size':<18} | {'Match Time (ms)':<15} | {'Ops per Second':<15}")
print("-" * 65)

for size in wm_sizes:
    current_wm = set(random.sample(fact_pool, min(size, len(fact_pool))))
    start_time = time.perf_counter()
    iterations = 50
    for _ in range(iterations):
        cs = matcher_large.compute_conflict_set(current_wm)
    end_time = time.perf_counter()

    avg_time_ms = ((end_time - start_time) / iterations) * 1000
    ops_per_sec = 1000 / avg_time_ms if avg_time_ms > 0 else 0
    print(f"{size:<10} | {len(cs):<18} | {avg_time_ms:<15.4f} | {ops_per_sec:<15.1f}")

print("\nWhy RETE Algorithm Exists:")
print("  1. Naive Matcher Complexity: O(|Rules| * |Conditions| * |WM|)")
print("     Re-evaluates EVERY rule against the ENTIRE Working Memory on every cycle.")
print("  2. RETE Algorithm Solution:")
print("     - Alpha Memories (1-input nodes) filter facts matching single conditions.")
print("     - Beta Memories (2-input nodes) join conditions across rules.")
print("     - Caches partial matches statefully across cycles.")
print("     - Time complexity reduces to O(dWM) — proportional ONLY to facts changed!")
